# Gradient Descent: Measuring Error and Learning from them

## Part 1 -- Measuring the Miss

The raw error is simply the difference between a prediction and its goal value; this gives a `delta` that tells us
- How far off we are
- Which way to go

Squaring that error does two things:
- Makes the error positive so that errors make sense (a negative error doesn't make much sense)
- Punishes larger errors

In [14]:
print(f"A delta of 2 gives a squared error of 4; i.e., 2 ** 2 == {2 ** 2}")

def delta(prediction, goal): return prediction - goal
def squared_error(prediction, goal): return (prediction - goal) ** 2

prediction = 2.0
goal = 1.5

print(f"Delta:{delta(prediction, goal):.4f}")
print(f"Squared error:{squared_error(prediction, goal):.4f}")

A delta of 2 gives a squared error of 4; i.e., 2 ** 2 == 4
Delta:0.5000
Squared error:0.2500


Mean squared error is just the mean of a number of predictions from one layer (whether there were multiple inputs or not, there were multiple outputs or predictions).

In [15]:
def mean_squared_error(predictions, goals): 
  total = 0
  for i in range(len(predictions)):
      total += squared_error(predictions[i], goals[i])
  return total / len(predictions)

predictions = [0.85, 1.0, 0.25, 1.1]
clean = [1.0, 1.0, 0.0, 1.0]

print(f"Mean squared error: {mean_squared_error(predictions, clean):.4f}")

Mean squared error: 0.0238


### NumPy Version

The main difference for `squared_error` is just working with NumPy types (`np.float64`) and the `np.square` function, but our `mean_squared_error` can easily become one line with NumPy by the power of broadcasting and elementwise functions.

In [24]:
import numpy as np

# The parameters should be type np.float64
def np_squared_error(prediction, goal): return np.square(prediction - goal)
def np_mean_squared_error(predictions, goals): return np.mean(np.square(predictions - goals))

# NumPy squared error
prediction = np.float64(2.0)
goal = np.float64(1.5)

print(f"Squared error:{np_squared_error(prediction, goal):.4f}")

# NumPy MSE
predictions = np.array([0.85, 1.0, 0.25, 1.1], dtype=float)
goals = np.array([1.0, 1.0, 0.0, 1.0], dtype=float)

print(f"Mean Squared Error:{np_mean_squared_error(predictions, goals):.4f}")

Squared error:0.2500
Mean Squared Error:0.0238


## Part 2 -- Gradient Descent: a Single Weight

Gradient descent is a full **predict-compare-learn** loop that updates the weight based on the error--specifically, based on the `delta`, or the "raw error" from Part 1...the difference between prediction and goal.

This implementation just takes
- One *input*,
- One *goal*,
- One *weight*, and
- Some number of *iterations*.

It then loops through that number of iterations and stores the error while learning the optimal weight for the input.

In [17]:
def gradient_descent(sensing, goal, weight, iterations):
  errors = [0] * iterations
  for i in range(iterations):
    prediction = sensing * weight # Predict
    error = squared_error(prediction, goal) # Compare
    errors[i] = error
    weight -= (prediction - goal) * sensing # Learn
  return errors

sensing = 0.65
goal = 1
weight = 0.5

print("Errors:")
gradient_descent(sensing, goal, weight, 20)

Errors:


[0.45562500000000006,
 0.15195378515625,
 0.050677537060766574,
 0.016901275344372264,
 0.005636680960319054,
 0.0018798683295224168,
 0.0006269478370727864,
 0.00020909102208775606,
 6.97331626851561e-05,
 2.3256445587766313e-05,
 7.756169956304764e-06,
 2.586731156489644e-06,
 8.626910077588964e-07,
 2.8771284290642527e-07,
 9.595403131455619e-08,
 3.2001269156139525e-08,
 1.0672623271489044e-08,
 3.559386564922173e-09,
 1.1870776655653952e-09,
 3.9589782069931525e-10]

### NumPy Version

As with Part 1's `squared_error`, the main difference between the above scratch version of basic gradient descent and the NumPy version is using `np.float64`.

In [25]:
import numpy as np

def np_gradient_descent(sensing, goal, weight, iterations):
  errors = [0] * iterations
  for i in range(iterations):
    prediction = sensing * weight # Predict
    errors[i] = np_squared_error(prediction, goal) # Compare
    weight -= (prediction - goal) * sensing # Learn
  return errors

sensing = np.float64(0.65)
goal = np.float64(1)
weight = np.float64(0.5)

print("Errors:")
np_gradient_descent(sensing, goal, weight, 20)

Errors:


[np.float64(0.45562500000000006),
 np.float64(0.15195378515625),
 np.float64(0.050677537060766574),
 np.float64(0.016901275344372264),
 np.float64(0.005636680960319054),
 np.float64(0.0018798683295224168),
 np.float64(0.0006269478370727864),
 np.float64(0.00020909102208775606),
 np.float64(6.97331626851561e-05),
 np.float64(2.3256445587766313e-05),
 np.float64(7.756169956304764e-06),
 np.float64(2.586731156489644e-06),
 np.float64(8.626910077588964e-07),
 np.float64(2.8771284290642527e-07),
 np.float64(9.595403131455619e-08),
 np.float64(3.2001269156139525e-08),
 np.float64(1.0672623271489044e-08),
 np.float64(3.559386564922173e-09),
 np.float64(1.1870776655653952e-09),
 np.float64(3.9589782069931525e-10)]

It's pretty much exactly the same loop; it just uses NumPy's special float types.

Notice how the `error` in the `# Compare` line isn't used to update the weight; does that mean it's not necessary to gradient descent? Although it may not be absolutely necessary to this particular implementation of the Gradient Descent algorithm, it is crucially the starting point from which you derive the engine behind gradient descent. The `# Learn` line uses the **derivative of the error with respect to the weight** to update the weight based on an error that's sensitive to the input.

## Part 3 -- Breaking, then Fixing with α

The basic gradient descent of Part 2 is fragile when given the wrong inputs or parameters. For example:

In [6]:
print("With an input of 2.0, " +
      "a goal of 0.8, and a " +
      "weight of 0.5, the error explodes:")
gradient_descent(2.0, 0.8, 0.5, 20)

With an input of 2.0, a goal of 0.8, and a weight of 0.5, the error explodes:


[0.03999999999999998,
 0.3599999999999998,
 3.2399999999999984,
 29.159999999999986,
 262.4399999999999,
 2361.959999999998,
 21257.639999999978,
 191318.75999999983,
 1721868.839999999,
 15496819.559999991,
 139471376.03999993,
 1255242384.3599997,
 11297181459.239996,
 101674633133.15994,
 915071698198.4395,
 8235645283785.954,
 74120807554073.56,
 667087267986662.1,
 6003785411879960.0,
 5.403406870691965e+16]

One way to fix this is to introduce a **learning rate** that prevents the weight update from diverging.

In [27]:
def gradient_descent_alpha(sensing, goal, weight, alpha, iterations):
  errors = [0] * iterations
  for i in range(iterations):
    errors[i] = squared_error(sensing * weight, goal)
    weight -= alpha * (sensing * weight - goal) * sensing
  return errors

gradient_descent_alpha(2.0, 0.8, 0.5, 0.1, 20)

[0.03999999999999998,
 0.0144,
 0.005183999999999993,
 0.0018662400000000014,
 0.0006718464000000028,
 0.00024186470400000033,
 8.70712934399997e-05,
 3.134566563839939e-05,
 1.1284439629823931e-05,
 4.062398266736526e-06,
 1.4624633760252567e-06,
 5.264868153690924e-07,
 1.8953525353291194e-07,
 6.82326912718715e-08,
 2.456376885786678e-08,
 8.842956788836216e-09,
 3.1834644439835434e-09,
 1.1460471998340758e-09,
 4.125769919393652e-10,
 1.485277170987127e-10]

Much better! The beautiful thing is that there was only one line different from this and Part 2's gradient descent--we multiplied the weight_delta ((sensing * weight - goal) * sensing) by an alpha value.

### NumPy Version

Again, the only real difference here is using `np.float64` values for the parameters. There is no real difference between the outputs of the two implementations.

In [28]:
def np_gradient_descent_alpha(input, goal, weight, alpha, iterations):
  errors = np.zeros(iterations)
  for i in range(iterations):
    errors[i] = squared_error(sensing * weight, goal)
    weight -= alpha * (sensing * weight - goal) * sensing
  return errors

np_gradient_descent_alpha(2.0, 0.8, 0.5, 0.1, 20)

array([0.225625  , 0.20696244, 0.18984356, 0.17414066, 0.15973663,
       0.14652402, 0.13440429, 0.12328705, 0.11308937, 0.10373519,
       0.09515474, 0.08728402, 0.08006433, 0.07344181, 0.06736708,
       0.06179482, 0.05668346, 0.05199489, 0.04769414, 0.04374912])

## Part 4: α Experimentation

Here we play around with the alpha values in search of one that will converge the quickest without diverging. Observe: